Setup: A bunch of packages that need to be installed before we get going.

sentence-transformers: For embedding the documents and queries.
numpy: For similarity comparisons.
scipy: For advanced similarity computations.
wikipedia-api: For loading a Wikipedia page as a knowledge base.
textwrap: For formatting output text.

**`Loading the Embedding Model`**
The gte-base-en-v1.5 model is an open-source English model provided by Alibaba's NLP team. It is part of the GTE (General Text Embeddings) family, designed for generating high-quality embeddings suitable for various natural language processing tasks. The model is optimized for capturing semantic meaning in English text, making it useful for tasks like sentence similarity, semantic search, and clustering. The trust_remote_code=True parameter allows the use of custom code associated with the model, ensuring that it functions as intended.

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-base-en-v1.5")


print(model.max_seq_length)

In [ ]:
from huggingface_hub import constants
#fetch Text Content from Wikipedia and Prepare It


from wikipediaapi import Wikipedia
wiki = Wikipedia('RAGBot/0.0', 'en')
doc = wiki.page('Hayao_Miyazaki').text
paragraphs = doc.split('\n\n')  # chunking

Chunking While there are a ton of chunking strategies available, many of them don't work as expected. It's best to review your knowledge base (KB) and determine which strategy suits it best. In this case, we'll chunk the document based on paragraphs. If you want to view how these chunks look, import the textwrap library, and enumerate over each paragraph to print them.

In [ ]:
import textwrap

for i , p in enumerate(paragraphs):
  wrapped_text = textwrap.fill(p,width =100)
  print("-----------------------------------------------------------------")
  print(wrapped_text)
  print("-----------------------------------------------------------------")



**Embed the Document**


These embeddings are dense vector representations of text that capture semantic meaning, allowing the model to understand and process text in a mathematical form.
We are normalizing the embeddings here.
What is normalization? It's a process that adjusts the values of the embeddings to have a unit norm (i.e., the length of the vector is 1).
Why normalize? Normalized embeddings ensure that the distance between vectors is primarily due to differences in direction rather than magnitude. This can improve the performance of models in tasks like similarity search, where you want to compare how "close" or "similar" different pieces of text are.
The result, docs_embed, is a collection of vector representations of your text data, where each vector corresponds to a paragraph in the paragraphs list.
The shape command gives the number of chunks and the dimension of each embedded vector. (Note that the size of the embedding vector depends on the type of embedding model.)



In [ ]:
docs_embed = model.encode(
    paragraphs,
    normalize_embeddings=True
)

print(docs_embed.shape)
docs_embed[0]

***embedd the query***

In [ ]:
query = "What was Studio Ghibli's first film?"
query_embed = model.encode(query, normalize_embeddings=True)
query_embed.shape

 Finding the Closest Paragraphs to the Query

One of the simplest ways to retrive of the most relevant chunks would be to compute the dot product of your document embedding and the query embedding.

a. Taking dot product¶

The dot product is a mathematical operation that multiplies corresponding elements of two vectors (or matrices) and sums the results. It is commonly used to measure the similarity between two vectors.


In [ ]:
import numpy as n 
similarities = n.dot(docs_embed , query_embed.T)
similarities.shape
print(similarities)

In [ ]:
top_3_idx = n.argsort(similarities, axis=0)[-3:][::-1].tolist()
print(top_3_idx)

most_similar_documents = [paragraphs[idx] for idx in top_3_idx]
print(most_similar_documents)
CONTEXT = ""
for i, p in enumerate(most_similar_documents):
  wrapped_text = textwrap.fill(p, width=100)

  print("-----------------------------------------------------------------")
  print(wrapped_text)
  print("-----------------------------------------------------------------")